# 供应商预期与发行人EPS口径核查
固定全部35次NVDA事件，数值接近仅作诊断，不选最接近值为actual，不读股价或事件收益。

In [1]:
from pathlib import Path
import os,json,hashlib
from decimal import Decimal
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.sec_eps_text import extract_company_nongaap_eps_text
from src.data.nvda_seasonal_eps import split_ratio_between_reports
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_consensus_basis_audit_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
f,fa=_source(root,p['filings'],['inputs/plan','inputs/web_text_bytes','analysis/verified_facts']);news,na=_source(root,p['consensus'],['analysis/events']);split,sa=_source(root,p['splits'],['inputs/split_ledger']);reports=f['analysis/verified_facts'];byq={x['quarter']:x for x in reports.to_dict('records')};bydate={x['release_date']:x for x in byq.values()};assert len(byq)==len(bydate)==48;cases={x['quarter']:x for x in f['inputs/plan']['cases']};non={}
for q,case in cases.items():
 payload=f['inputs/web_text_bytes'][case['file']];assert hashlib.sha256(payload).hexdigest()==case['sha256'];non[q]=extract_company_nongaap_eps_text(payload.decode(),quarter=q,accession=case['accession'])
news=pd.DataFrame(news['analysis/events']);news=news[news.symbol.eq('NVDA')].copy();assert len(news)==35 and not news.report_date.duplicated().any();rows=[]
for row in news.to_dict('records'):
 date=str(pd.Timestamp(row['report_date']).date());now=bydate[date];q=now['quarter'];previous=f'FY{int(q[2:6])-1}Q{q[-1]}';old=byq[previous];factor=split_ratio_between_reports(old,now,split['inputs/split_ledger']);prior=float(row['prior_year_eps']);assert np.isfinite(prior)
 values={'original_gaap_on_current_split_basis':float(Decimal(old['current_eps'])/factor),'original_nongaap_on_current_split_basis':float(Decimal(non[previous]['current_eps'])/factor),'current_comparative_gaap':float(now['prior_year_comparative_eps']),'current_comparative_nongaap':float(non[q]['prior_year_comparative_eps'])}
 record={'quarter':q,'prior_quarter':previous,'report_date':date,'newsletter_prior_year_eps':prior,'newsletter_consensus_eps':float(row['consensus_eps']),'split_ratio':str(factor),'newsletter_url':row['url'],'issuer_url':now['source_url'],'consensus_basis_certified':False,**values}
 for key,value in values.items():record[key+'_within_half_cent']=bool(abs(prior-value)<=p['cent_proximity']+1e-12)
 rows.append(record)
table=pd.DataFrame(rows).sort_values('report_date');checks=[x for x in table if x.endswith('_within_half_cent')];counts={x:int(table[x].sum()) for x in checks};counts['neither_current_comparative']=int((~(table.current_comparative_gaap_within_half_cent|table.current_comparative_nongaap_within_half_cent)).sum());policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/basis_comparison','analysis/issuer_non_gaap']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'filings':fa,'consensus':na,'splits':sa},parser=(root/'src/data/sec_eps_text.py').read_text());rec.save_objects(artifact_path='analysis',basis_comparison=table,issuer_non_gaap=non);rec.log_metrics(compared_events=len(table),new_fits=0,new_backtests=0)
summary={'experiment':card.name,'decision':'needs_vendor_comparable_actuals_and_vintage','compared_events':len(table),'numeric_proximity_counts':counts,'numeric_match_is_basis_certification':False,'new_fits':0,'new_backtests':0,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-14','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope']};rec.save_objects(artifact_path='analysis',summary=summary);a=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in a.rglob('*')));table.to_csv(card/'basis_comparison.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(counts);print(table[['report_date','newsletter_prior_year_eps',*values]].head().to_string(index=False))


# EPS一致预期口径核查：不能直接指定GAAP或公司非GAAP为actual

固定读取已有105条Refinitiv简报预期中的全部35次NVDA事件，逐日期绑定48季度原始SEC申报文本；未查看股价或事件后收益，0拟合、0回测。来源并非只剩早期12条，已扩展的105条记录及PDF都在父Recorder，未重新下载或修数。

LSEG官方说明：I/B/E/S actuals按多数分析师采用的经营/可比口径调整；comparable actual、reported GAAP actual、restated actual和go-forward actual是不同概念。公司自行定义的非GAAP数字不能自动代替供应商可比actual。依据：https://www.lseg.com/en/data-catalogue/company-data/ibes-estimates/actuals 和 https://lipperalpha.refinitiv.com/2023/03/earnings-insight-how-are-companies-actually-reporting-earnings/ 。

本轮对简报“去年EPS”做事后数值核对，半美分仅为预登记的描述性接近尺度，不是口径认证：35例中5例接近拆股换算后的原报告GAAP，30例接近原报告公司非GAAP；比较当期报告里的去年栏时，GAAP5例、公司非GAAP28例，4例两者都不接近。各计数允许重叠，不加总成35，也不按最近距离选actual。所有比较和差异完整保留。

|简报日期|简报去年EPS|同股份口径原GAAP|同股份口径原公司非GAAP|当期比较非GAAP|
|---|---:|---:|---:|---:|
|2019-02-14|1.57|1.78|1.72|1.72|
|2022-05-25|0.92|0.7575|0.915|0.91|
|2026-05-20|0.96|0.76|0.81|0.78|
|2026-08-26|1.05|1.08|1.05|1.01|

这些差异不能全归为数据错误。2022例涉及拆股和显示精度；2026-05简报0.96与公司FY2026Q1原报告单列的“剔除H20费用及相关税项”0.96相同，而普通非GAAP为0.81（原始附件：https://www.sec.gov/Archives/edgar/data/1045810/000104581025000115/q1fy26pr.htm）。这能反证统一取公司非GAAP的简单规则，不能仅凭数字相同证明供应商的完整调整政策。2019例尚未查清，不推导补数或挑可比值。

更重要的是，本轮简报的“去年EPS”只是在当前简报日期看到的后来历史值，可能反映重述或后来口径；禁止把它回填为去年公告时已知的供应商actual。当前预期与当前可比actual、历史预期版本、提前预警/修订及发布时间需要分别绑定。既有原公司报告值保持不变。

决定needs_vendor_comparable_actuals_and_vintage，不生成surprise、不重新训练旧供应商EPS路线。下一具体信息需求是同期可比actual及其口径/生效版本，不能用GAAP、公司非GAAP或次年简报去年值替代。105份预期PDF的历史公开时间认证也仍缺失；数字可以核对不等于首发或生产资格。此来源仍可做进一步限定核查，不能把“尚无成功模型”直接当成全部研究已无路可走。

1项解析合同测试通过（新增非GAAP列读取及非供应商口径标记），48份非GAAP结果在增加表头检查后逐值不变，NAS冷恢复逐对象哈希通过。生产及既有收益不变，稳定净SOXX超额未达成。
